# Notebook 6 — Interprétabilité
Coefficients, Feature Importance, SHAP


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../health_lifestyle_dataset.csv')
df['gender_enc'] = (df['gender'] == 'Male').astype(int)
df['hypertension'] = ((df['systolic_bp'] >= 140) | (df['diastolic_bp'] >= 90)).astype(int)
df['bmi_cat'] = pd.cut(df['bmi'], bins=[0,18.5,25,30,100], labels=[0,1,2,3]).astype(int)
df['lifestyle_score'] = (
    df['daily_steps']/df['daily_steps'].max() + df['sleep_hours']/df['sleep_hours'].max() +
    df['water_intake_l']/df['water_intake_l'].max() - df['smoker'] - df['alcohol'] -
    df['bmi']/df['bmi'].max() - df['calories_consumed']/df['calories_consumed'].max())
features = ['age','bmi','daily_steps','sleep_hours','water_intake_l','calories_consumed',
            'smoker','alcohol','resting_hr','systolic_bp','diastolic_bp',
            'family_history','gender_enc','bmi_cat','hypertension','lifestyle_score']
X = df[features]; y = df['disease_risk']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train); X_te_sc = scaler.transform(X_test)

## 1. Coefficients Régression Logistique

In [ ]:
lr = LogisticRegression(C=0.07, penalty='l1', solver='liblinear', class_weight='balanced', max_iter=1000)
lr.fit(X_tr_sc, y_train)
coef = pd.Series(lr.coef_[0], index=features).sort_values()
fig, ax = plt.subplots(figsize=(8,6))
coef.plot(kind='barh', ax=ax, color=['tomato' if x<0 else 'steelblue' for x in coef])
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Coefficients Régression Logistique (L1)', fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/lr_coefficients.png', dpi=150)
plt.show()

## 2. Feature Importance Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=8, max_samples=0.5, class_weight='balanced', n_jobs=-1, random_state=42)
rf.fit(X_train.values, y_train)
fi_rf = pd.Series(rf.feature_importances_, index=features).sort_values()
fig, ax = plt.subplots(figsize=(8,6))
fi_rf.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importance — Random Forest', fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/rf_feature_importance.png', dpi=150)
plt.show()

## 3. SHAP Values (XGBoost)

In [ ]:
xgb = XGBClassifier(scale_pos_weight=3, n_estimators=200, max_depth=3, learning_rate=0.05, n_jobs=-1, random_state=42, verbosity=0, eval_metric='logloss')
xgb.fit(X_train.values, y_train)
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test.values[:500])
shap.summary_plot(shap_values, X_test.values[:500], feature_names=features, show=False)
plt.title('SHAP Summary Plot — XGBoost', fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Tableau de convergence des méthodes

In [ ]:
shap_mean = pd.Series(np.abs(shap_values).mean(axis=0), index=features).sort_values(ascending=False)
print('Top 5 variables par méthode :')
print(f'LogReg (coef abs) : {list(coef.abs().sort_values(ascending=False).head(5).index)}')
print(f'RF importance     : {list(fi_rf.sort_values(ascending=False).head(5).index)}')
print(f'SHAP XGBoost      : {list(shap_mean.head(5).index)}')